In [1]:
import sys, os
from google.colab import drive

drive.mount("/content/drive")

REPO_DIR = "/content/drive/MyDrive/Duke/CS 590 RL/Project/balatro-rl-agent"
os.chdir(REPO_DIR)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import numpy as np
import torch
from functools import partial
from torch.distributions import Categorical

from gymnasium.vector import AsyncVectorEnv

from balatro_gym.constants import MAX_HAND_SIZE
from cs590_env.combat_env import (
    make_pooled_combat_env,
    VecRolloutBuffer,
    compute_gae_vectorized,
    dict_to_tensors,
    get_card_mask,
    mask_logits,
)
from balatro_gym.save_injection import load_snapshot_pool

%run cs590_src/BalatroPPO.ipynb

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cfg = PPOConfig()

snapshot_pool = load_snapshot_pool()

agent = CombatPPOAgent(
    d_model=cfg.d_model, nhead=cfg.nhead,
    dim_ff=cfg.dim_ff, dropout=cfg.dropout,
).to(device)

optimizer = torch.optim.Adam(agent.parameters(), lr=cfg.lr, eps=1e-5)
buffer    = VecRolloutBuffer(cfg.rollout_steps, cfg.num_envs, device)

BASE_SEED = 12345
vec_env = AsyncVectorEnv([
    partial(make_pooled_combat_env, snapshot_pool, pool_seed=BASE_SEED + i)
    for i in range(cfg.num_envs)
])

In [ ]:
obs_np, _ = vec_env.reset()
ep_returns: list[float] = []
ep_lengths: list[int]   = []
running_rewards = np.zeros(cfg.num_envs)
running_lens    = np.zeros(cfg.num_envs, dtype=int)

for iteration in range(1, cfg.max_iterations + 1):

    # ── Phase 1: Parallel rollout collection ─────────────────────
    agent.eval()
    for t in range(cfg.rollout_steps):
        obs_t = dict_to_tensors(obs_np, device)
        card_mask = get_card_mask(obs_t)

        with torch.no_grad():
            sel_logits, exec_logits, value = agent(obs_t)

        sel_logits = mask_logits(sel_logits, card_mask)

        sel_dist  = Categorical(logits=sel_logits)
        exec_dist = Categorical(logits=exec_logits)

        card_sels  = sel_dist.sample()
        executions = exec_dist.sample()

        log_probs = (sel_dist.log_prob(card_sels).sum(dim=-1)
                     + exec_dist.log_prob(executions))

        actions_np = np.concatenate([
            card_sels.cpu().numpy(),
            executions.cpu().numpy()[:, None],
        ], axis=-1)

        next_obs_np, rewards_np, terms, truncs, infos = vec_env.step(actions_np)
        dones_np = np.logical_or(terms, truncs)

        buffer.store_step(
            t, obs_t, card_sels, executions, log_probs,
            value.squeeze(-1), rewards_np, dones_np,
        )

        running_rewards += rewards_np
        running_lens    += 1
        for i in np.where(dones_np)[0]:
            ep_returns.append(running_rewards[i])
            ep_lengths.append(running_lens[i])
            running_rewards[i] = 0.0
            running_lens[i]    = 0

        obs_np = next_obs_np

    # ── Phase 2: Per-env GAE + PPO update ────────────────────────
    with torch.no_grad():
        _, _, next_vals = agent(dict_to_tensors(obs_np, device))
        next_vals = next_vals.squeeze(-1)

    advantages, returns = compute_gae_vectorized(
        buffer.rewards, buffer.values, next_vals, buffer.dones,
        cfg.gamma, cfg.gae_lambda,
    )

    flat_obs, flat_card_sels, flat_execs, flat_lp = buffer.flatten()
    flat_adv = advantages.reshape(-1)
    flat_ret = returns.reshape(-1)

    agent.train()
    losses = ppo_update(
        agent, optimizer,
        flat_obs, flat_card_sels, flat_execs, flat_lp,
        flat_adv, flat_ret, cfg,
    )

    # ── Logging ──────────────────────────────────────────────────
    if iteration % 5 == 0 or iteration == 1:
        recent_r   = np.mean(ep_returns[-50:]) if ep_returns else 0.0
        recent_len = np.mean(ep_lengths[-50:]) if ep_lengths else 0.0
        print(f"[iter {iteration:4d}]  "
              f"reward={recent_r:+.2f}  len={recent_len:.1f}  "
              f"pg={losses['pg_loss']:.4f}  "
              f"vf={losses['value_loss']:.4f}  "
              f"ent={losses['entropy']:.4f}  "
              f"episodes={len(ep_returns)}")

In [ ]:
vec_env.close()

checkpoint_path = "checkpoints/combat_ppo.pt"
os.makedirs(os.path.dirname(checkpoint_path), exist_ok=True)
torch.save({
    "model_state_dict": agent.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "config": cfg,
}, checkpoint_path)
print(f"Checkpoint saved to {checkpoint_path}")